# Acoustic Teleoperation Analysis
**Configurations:**
- `FHSS` — FHSS
- `BPSK` — BPSK

**Pipeline:** log → clean → match → metrics → statistical tests → comparative plots

In [7]:
%pip install pandas numpy plotly scipy statsmodels kaleido nbformat

Note: you may need to restart the kernel to use updated packages.


## 0. Configuration

In [8]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import re
import csv
import io
import os
import warnings
from scipy import stats
from scipy.stats import mannwhitneyu, shapiro, kruskal
from itertools import combinations
from statsmodels.stats.proportion import proportion_confint

warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR    = "./"           # Root directory for input data
FIGURES_DIR = "./figures"    # Output directory for exported PDF figures

CONFIGS = {
    "Lab, Trial 1: FHSS\n": f"{BASE_DIR}/lab_1/janus",
    "Lab, Trial 2: FHSS\n": f"{BASE_DIR}/lab_2/janus",
    "Lab, Trial 1: BPSK\n": f"{BASE_DIR}/lab_1/flex",
    "Lab, Trial 2: BPSK\n": f"{BASE_DIR}/lab_2/flex",
    "River, Trial 1: FHSS\n": f"{BASE_DIR}/river_1/janus",
    "River, Trial 2: FHSS\n": f"{BASE_DIR}/river_2/janus",
    "River, Trial 1: BPSK\n": f"{BASE_DIR}/river_1/flex",
    "River, Trial 2: BPSK\n": f"{BASE_DIR}/river_2/flex",

}

MAX_FORCE      = 50       # Force on the config file
MAX_DELAY_MS   = 15000    # matching window [ms] — must exceed max expected latency
ALPHA          = 0.05     # significance level for statistical tests

COLORS = [
    "#1f77b4",  # Lab, FHSS (blue)
    "#4fa3d1",  # Lab, FHSS (lighter blue)

    "#9467bd",  # Lab, BPSK (purple)
    "#c5a5e0",  # Lab, BPSK (lighter purple)

    "#2ca02c",  # River, FHSS (green)
    "#6fcf6f",  # River, FHSS (lighter green)

    "#ff7f0e",  # River, BPSK (orange)
    "#ffb570",  # River, BPSK (lighter orange)
]

# ── PDF export setup ───────────────────────────────────────────────────────
pio.kaleido.scope.default_format = "pdf"
os.makedirs(FIGURES_DIR, exist_ok=True)

def save_fig(fig, name: str):
    """Export a Plotly figure as PDF to FIGURES_DIR."""
    path = os.path.join(FIGURES_DIR, f"{name}.pdf")
    fig.write_image(path)
    print(f"  ✔ Saved: {path}")

print("Configuration loaded.")

Configuration loaded.


## 1. Preprocessing Functions

In [9]:
def log_to_df(log_path: str) -> pd.DataFrame:
    """Parse receiver .log file → DataFrame"""
    pattern = re.compile(r"ROS_TIME_NS=(\d+).*?values=([-\d,]+)")
    rows = []
    with open(log_path, "r") as f:
        for line in f:
            m = pattern.search(line)
            if m:
                ros_time_ns = int(m.group(1))
                time_ms = ros_time_ns // 1_000_000
                values = list(map(int, m.group(2).split(",")))
                rows.append([time_ms] + values)
    cols = ["time"] + [f"value_{i}" for i in range(1, len(rows[0]))]
    return pd.DataFrame(rows, columns=cols)


def clean_controller_df(df: pd.DataFrame) -> pd.DataFrame:
    """Drop trigger columns, truncate floats, remove consecutive duplicates."""
    df = df.drop(columns=[c for c in ["leftT", "rightT"] if c in df.columns])
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(0).astype(float)
    sensor_cols = [c for c in df.columns if c != "epoch"]
    mask = df[sensor_cols].ne(df[sensor_cols].shift()).any(axis=1)
    return pd.concat([df.iloc[[0]], df[mask]]).drop_duplicates().reset_index(drop=True)


def match(controller_df: pd.DataFrame,
          receiver_df: pd.DataFrame,
          max_delay_ms: int = MAX_DELAY_MS) -> pd.DataFrame:
    """
    Match sent commands (controller) to received commands (receiver).
    Returns a DataFrame with columns:
      controller_time, command_time, status
    where status ∈ {matched, lost, noSended}

    NOTE: 'noSended' medians a packet was received but could not be matched
    to any sent command within the time window.  Investigate these carefully;
    they may indicate clock drift, retransmissions, or logging artefacts.
    """
    f1 = controller_df.sort_values("epoch").reset_index(drop=True)
    f2 = receiver_df.sort_values("time").reset_index(drop=True)

    # Normalise receiver values to controller scale
    r = pd.DataFrame()
    r["time"]   = pd.to_numeric(f2["time"], errors="coerce")
    r["leftV"]  =  (pd.to_numeric(f2.iloc[:, 1], errors="coerce") / MAX_FORCE)
    r["leftH"]  =  (pd.to_numeric(f2.iloc[:, 2], errors="coerce") / MAX_FORCE)
    r["rightH"] =  (pd.to_numeric(f2.iloc[:, 3], errors="coerce") / MAX_FORCE)
    r["rightV"] =  (pd.to_numeric(f2.iloc[:, 4], errors="coerce") / MAX_FORCE)

    for col in ["epoch", "leftH", "leftV", "rightH", "rightV"]:
        f1[col] = pd.to_numeric(f1[col], errors="coerce")

    results, used = [], set()
    j = 0
    for i in range(len(f1)):
        row1 = f1.iloc[i]
        t_ctrl = row1["epoch"]
        matched = False
        while j < len(r) and r.iloc[j]["time"] < t_ctrl:
            j += 1
        k = j
        while k < len(r) and r.iloc[k]["time"] <= t_ctrl + max_delay_ms:
            if k not in used:
                row2 = r.iloc[k]
                if (np.isclose(row1["leftH"],  row2["leftH"]) and
                    np.isclose(row1["leftV"],  row2["leftV"]) and
                    np.isclose(row1["rightH"], row2["rightH"]) and
                    np.isclose(row1["rightV"], row2["rightV"])):
                    results.append({"controller_time": t_ctrl,
                                    "command_time": row2["time"],
                                    "status": "matched"})
                    used.add(k)
                    matched = True
                    break
            k += 1
        if not matched:
            results.append({"controller_time": t_ctrl,
                            "command_time": None,
                            "status": "lost"})

    for idx in range(len(r)):
        if idx not in used:
            results.append({"controller_time": None,
                            "command_time": r.iloc[idx]["time"],
                            "status": "noSended"})

    out = pd.DataFrame(results)
    out["sort_time"] = out["controller_time"].fillna(out["command_time"])
    out = out.sort_values("sort_time").drop(columns=["sort_time"]).reset_index(drop=True)
    return out


print("Functions defined.")

Functions defined.


## 2. Load and Process All Configurations

In [10]:
matched_data = {}   # label → matched DataFrame
raw_stats    = {}   # label → dict of counts

for label, base in CONFIGS.items():
    print(f"\n{'='*55}")
    print(f"Processing: {label.replace(chr(10), ' ')}")
    print('='*55)

    try:
        rec_df  = log_to_df(f"{base}/received_log.log")
        ctrl_df = pd.read_csv(f"{base}/controller_log.csv")
    except FileNotFoundError as e:
        print(f"  [SKIP] File not found: {e}")
        continue

    ctrl_clean = clean_controller_df(ctrl_df)
    matched    = match(ctrl_clean, rec_df)

    if "matched" in matched["status"].values:
        first_matched_idx = matched.index[matched["status"] == "matched"][0]
        matched = matched.loc[first_matched_idx + 1:].reset_index(drop=True)

    n_total    = len(matched[matched["status"].isin(["matched", "lost"])])
    n_matched  = len(matched[matched["status"] == "matched"])
    n_lost     = len(matched[matched["status"] == "lost"])
    n_nosended = len(matched[matched["status"] == "noSended"])

    print(f"  Sent commands  : {n_total}")
    print(f"  Matched        : {n_matched}")
    print(f"  Lost           : {n_lost}")
    print(f"  noSended       : {n_nosended}  ← investigate these")

    if n_nosended > 0:
        print(f"[WARNING]: {n_nosended} received packets could not be matched "
              f"to any sent command. Possible causes: clock drift artefact, "
              f"duplicate transmission, or logging error. Exclude from PDR "
              f"calculation until cause is identified.")

    if n_total < 30:
        print(f"[WARNING]: sample size {n_total} < 30. "
              f"Statistical power is limited.")

    matched_data[label] = matched
    raw_stats[label] = {
        "n_sent": n_total, "n_matched": n_matched,
        "n_lost": n_lost,  "n_nosended": n_nosended
    }

print("\n[OK] All configurations processed.")


Processing: Lab, Trial 1: FHSS 
  Sent commands  : 195
  Matched        : 195
  Lost           : 0
  noSended       : 0  ← investigate these

Processing: Lab, Trial 2: FHSS 
  Sent commands  : 224
  Matched        : 222
  Lost           : 2
  noSended       : 0  ← investigate these

Processing: Lab, Trial 1: BPSK 
  Sent commands  : 135
  Matched        : 135
  Lost           : 0
  noSended       : 0  ← investigate these

Processing: Lab, Trial 2: BPSK 
  Sent commands  : 370
  Matched        : 369
  Lost           : 1
  noSended       : 0  ← investigate these

Processing: River, Trial 1: FHSS 
  Sent commands  : 115
  Matched        : 98
  Lost           : 17
  noSended       : 0  ← investigate these

Processing: River, Trial 2: FHSS 
  Sent commands  : 122
  Matched        : 113
  Lost           : 9
  noSended       : 0  ← investigate these

Processing: River, Trial 1: BPSK 
  Sent commands  : 83
  Matched        : 78
  Lost           : 5
  noSended       : 0  ← investigate these

P

## 3. Compute Per-Configuration Metrics

In [ ]:
# ── Outlier thresholds ────────────────────────────────────────────────────────
LATENCY_OUTLIER_MS  = 5_000 
JITTER_OUTLIER_MS   = 1_000
# ─────────────────────────────────────────────────────────────────────────────

metrics = {}   # label → dict of metrics

for label, df in matched_data.items():
    s     = raw_stats[label]
    df_m  = df[df["status"] == "matched"].copy()
    df_m["latency_ms"] = df_m["command_time"] - df_m["controller_time"]

    # ── Sanity check: negative latencies indicate clock issue ─────────────────
    neg = (df_m["latency_ms"] < 0).sum()
    if neg > 0:
        print(f"[{label.replace(chr(10), ' ')}] {neg} negative latencies detected. "
              f"Check clock synchronisation.")

    lat_raw = df_m["latency_ms"].dropna()

    # ── Outlier detection (latency) ───────────────────────────────────────────
    outlier_mask        = lat_raw > LATENCY_OUTLIER_MS
    n_outliers_latency  = int(outlier_mask.sum())
    lat_outlier_vals    = lat_raw[outlier_mask].values

    outlier_indices     = np.where(outlier_mask.values)[0]
    lat                 = lat_raw[~outlier_mask]

    lat_full_len        = len(lat_raw)

    if n_outliers_latency > 0:
        print(f"[{label.replace(chr(10), ' ')}] {n_outliers_latency} latency "
              f"outlier(s) removed (> {LATENCY_OUTLIER_MS} ms): "
              f"{lat_outlier_vals.tolist()} ms")

    # ── Jitter (computed on cleaned latency, then filtered separately) ────────
    jitter_raw          = lat.diff().abs().dropna()
    outlier_mask_jit    = jitter_raw > JITTER_OUTLIER_MS
    n_outliers_jitter   = int(outlier_mask_jit.sum())
    jitter_outlier_vals = jitter_raw[outlier_mask_jit].values
    jitter              = jitter_raw[~outlier_mask_jit]         # cleaned series

    if n_outliers_jitter > 0:
        print(f"[{label.replace(chr(10), ' ')}] {n_outliers_jitter} jitter "
              f"outlier(s) removed (> {JITTER_OUTLIER_MS} ms): "
              f"{jitter_outlier_vals.tolist()} ms")

    # ── PDR (uses original sent/matched counts — outliers do NOT affect PDR) ──
    pdr = s["n_matched"] / s["n_sent"] if s["n_sent"] > 0 else np.nan

    # ── Normality test (on cleaned latency) ───────────────────────────────────
    if len(lat) >= 3:
        sw_stat, sw_p = shapiro(lat)
        normal        = sw_p > ALPHA
    else:
        sw_stat, sw_p, normal = np.nan, np.nan, False

    n        = len(lat)
    mean_lat = lat.mean()
    std_lat  = lat.std(ddof=1)

    # ── 95 % CI on mean ───────────────────────────────────────────────────────
    if normal and n >= 30:
        se       = std_lat / np.sqrt(n)
        ci_low   = mean_lat - 1.96 * se
        ci_high  = mean_lat + 1.96 * se
        ci_method = "Normal (z=1.96)"
    else:
        rng = np.random.default_rng(42)
        boots_mean = [
            rng.choice(lat, size=n, replace=True).mean()
            for _ in range(5000)
        ]
        ci_low, ci_high = np.percentile(boots_mean, [2.5, 97.5])
        ci_method = "Bootstrap mean (n=5000)"

    # ── 95 % CI on median ─────────────────────────────────────────────────────
    rng    = np.random.default_rng(42)
    n_boot = 5000
    if n >= 2:
        boots_median = [
            np.median(rng.choice(lat, size=n, replace=True))
            for _ in range(n_boot)
        ]
        ci_median_low, ci_median_high = np.percentile(boots_median, [2.5, 97.5])
    else:
        ci_median_low, ci_median_high = np.nan, np.nan

    # ── Store metrics ─────────────────────────────────────────────────────────
    metrics[label] = {
        # Delivery
        "pdr":          pdr,
        "packet_loss":  1.0 - pdr,
        "n_sent":       s["n_sent"],
        "n_matched":    s["n_matched"],
        "n_lost":       s["n_lost"],

        # Outliers (reported in table, excluded from all stats below)
        "n_outliers_latency":     n_outliers_latency,
        "outlier_latency_vals_ms": lat_outlier_vals.tolist(),
        "n_outliers_jitter":      n_outliers_jitter,
        "outlier_jitter_vals_ms": jitter_outlier_vals.tolist(),
        "outlier_indices":    outlier_indices,
        "lat_full_len":       lat_full_len,

        # Mean stats  (cleaned)
        "lat_mean":     mean_lat,
        "lat_std":      std_lat,
        "ci_low":       ci_low,
        "ci_high":      ci_high,
        "ci_method":    ci_method,

        # Median stats (cleaned)
        "lat_median":         lat.median(),
        "lat_median_ci_low":  ci_median_low,
        "lat_median_ci_high": ci_median_high,

        # Distribution (cleaned)
        "lat_p5":   lat.quantile(0.05),
        "lat_p95":  lat.quantile(0.95),
        "lat_min":  lat.min(),
        "lat_max":  lat.max(),

        # Jitter (cleaned)
        "jitter_median": jitter.median(),
        "jitter_min":    jitter.min(),
        "jitter_max":    jitter.max(),

        # Normality
        "shapiro_p": sw_p,
        "is_normal": normal,

        # Raw arrays (cleaned, for plots)
        "latency_vals": lat.values,
        "jitter_vals":  jitter.values,
    }

print("Metrics computed.")

AttributeError: 'int' object has no attribute 'sum'

## 4. Summary Table

In [ ]:
rows = []
for label, m in metrics.items():
    rows.append({
        "Config":               label.replace("\n", " "),
        "N sent":               m["n_sent"],
        "N matched":            m["n_matched"],
        "N lost":               m["n_lost"],
        "PDR (%)":              f"{m['pdr']*100:.1f}",
        "Loss (%)":             f"{m['packet_loss']*100:.1f}",

        # Outliers (reported but excluded from stats)
        "Outliers lat/jit":     f"{m['n_outliers_latency']} / {m['n_outliers_jitter']}",

        # Mean (cleaned)
        "Lat mean (ms)":        f"{m['lat_mean']:.1f}",
        "Lat std (ms)":         f"{m['lat_std']:.1f}",
        "CI mean 95%":          f"[{m['ci_low']:.1f}, {m['ci_high']:.1f}]",

        # Median (cleaned)
        "Lat median (ms)":      f"{m['lat_median']:.1f}",
        "CI median 95%":        f"[{m['lat_median_ci_low']:.1f}, {m['lat_median_ci_high']:.1f}]",

        # Distribution (cleaned)
        "Lat p5 (ms)":          f"{m['lat_p5']:.1f}",
        "Lat p95 (ms)":         f"{m['lat_p95']:.1f}",
        "Lat min (ms)":         f"{m['lat_min']:.1f}",
        "Lat max (ms)":         f"{m['lat_max']:.1f}",

        # Jitter (cleaned)
        "Jitter median (ms)":   f"{m['jitter_median']:.1f}",
        "Jitter min (ms)":      f"{m['jitter_min']:.1f}",
        "Jitter max (ms)":      f"{m['jitter_max']:.1f}",
    })

summary = pd.DataFrame(rows).set_index("Config")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)
display(summary.T)

Config,"Lab, Trial 1: FHSS","Lab, Trial 2: FHSS","Lab, Trial 1: BPSK","Lab, Trial 2: BPSK","River, Trial 1: FHSS","River, Trial 2: FHSS","River, Trial 1: BPSK","River, Trial 2: BPSK"
N sent,195,224,135,370,115,122,83,128
N matched,195,222,135,369,98,113,78,117
N lost,0,2,0,1,17,9,5,11
PDR (%),100.0,99.1,100.0,99.7,85.2,92.6,94.0,91.4
Loss (%),0.0,0.9,0.0,0.3,14.8,7.4,6.0,8.6
Outliers lat/jit,0 / 0,0 / 0,0 / 0,0 / 0,6 / 11,3 / 3,0 / 2,6 / 4
Lat mean (ms),2830.9,3413.9,2223.6,2310.9,3043.6,3672.0,2027.7,3044.2
Lat std (ms),179.2,185.6,63.5,82.4,456.9,258.8,419.1,339.3
CI mean 95%,"[2806.7, 2855.9]","[3390.0, 3437.6]","[2212.9, 2234.3]","[2302.6, 2319.5]","[2955.6, 3138.5]","[3622.7, 3721.6]","[1937.9, 2123.3]","[2983.7, 3108.4]"
Lat median (ms),2867.0,3400.0,2221.0,2305.0,2963.5,3694.0,1845.5,2986.0


## 5. Statistical Tests

**Decision rule:** If any distribution is non-normal (Shapiro-Wilk p ≤ 0.05), use **Mann–Whitney U** (non-parametric) for pairwise comparison. If all are normal, use **Welch's t-test**. For global comparison across all 3, use **Kruskal–Wallis**.

In [ ]:
labels = list(metrics.keys())
lat_arrays = [metrics[l]["latency_vals"] for l in labels]
all_normal = all(metrics[l]["is_normal"] for l in labels)

print("Normality check (Shapiro-Wilk):")
for l in labels:
    flag = "normal" if metrics[l]["is_normal"] else "non-normal"
    print(f"  {l.replace(chr(10),' '):30s}  p={metrics[l]['shapiro_p']:.4f}  {flag}")

test_name = "Welch's t-test" if all_normal else "Mann–Whitney U"
print(f"\n→ Using: {test_name} for pairwise comparisons")

# Global test
if len(lat_arrays) >= 3:
    kw_stat, kw_p = kruskal(*lat_arrays)
    print(f"\nKruskal–Wallis test (all groups):  H={kw_stat:.3f},  p={kw_p:.4f}")
    if kw_p < ALPHA:
        print("  → At least one group differs significantly (p < 0.05)")
    else:
        print("  → No significant difference detected across groups")

# Pairwise
print(f"\nPairwise comparisons ({test_name}):")
print(f"{'Pair':50s}  {'Statistic':>12s}  {'p-value':>10s}  Result")
print("-" * 90)
for l1, l2 in combinations(labels, 2):
    a1 = metrics[l1]["latency_vals"]
    a2 = metrics[l2]["latency_vals"]
    if all_normal:
        stat, p = stats.ttest_ind(a1, a2, equal_var=False)
    else:
        stat, p = mannwhitneyu(a1, a2, alternative="two-sided")
    sig = "significant" if p < ALPHA else "not significant"
    pair = f"{l1.replace(chr(10),' ')} vs {l2.replace(chr(10),' ')}"
    print(f"{pair:50s}  {stat:12.3f}  {p:10.4f}  {sig}")


    # ── Trial consistency check (within same modulation × environment) ────────────
consistency_pairs = [
    ("Lab, Trial 1: FHSS\n",  "Lab, Trial 2: FHSS\n",  "Lab – FHSS"),
    ("Lab, Trial 1: BPSK\n",      "Lab, Trial 2: BPSK\n",      "Lab – BPSK"),
    ("River, Trial 1: FHSS\n","River, Trial 2: FHSS\n","River – FHSS"),
    ("River, Trial 1: BPSK\n",    "River, Trial 2: BPSK\n",    "River – BPSK"),
]

print("Trial-to-Trial Consistency (Mann–Whitney U, two-sided):")
print(f"{'Condition':25s}  {'U':>10s}  {'p-value':>10s}  {'Consistent?':>12s}")
print("-" * 65)
pooled_latencies = {}
for l1, l2, name in consistency_pairs:
    a1 = metrics[l1]["latency_vals"]
    a2 = metrics[l2]["latency_vals"]
    u, p = mannwhitneyu(a1, a2, alternative="two-sided")
    consistent = "YES (p>0.05)" if p > ALPHA else "NO — investigate"
    print(f"{name:25s}  {u:10.1f}  {p:10.4f}  {consistent:>12s}")
    pooled_latencies[name] = np.concatenate([a1, a2])

print("\n→ Conditions with p > 0.05 can be pooled without bias.")

Normality check (Shapiro-Wilk):
  Lab, Trial 1: FHSS              p=0.0000  non-normal
  Lab, Trial 2: FHSS              p=0.0000  non-normal
  Lab, Trial 1: BPSK              p=0.4119  normal
  Lab, Trial 2: BPSK              p=0.0000  non-normal
  River, Trial 1: FHSS            p=0.0000  non-normal
  River, Trial 2: FHSS            p=0.0005  non-normal
  River, Trial 1: BPSK            p=0.0000  non-normal
  River, Trial 2: BPSK            p=0.0000  non-normal

→ Using: Mann–Whitney U for pairwise comparisons

Kruskal–Wallis test (all groups):  H=1122.927,  p=0.0000
  → At least one group differs significantly (p < 0.05)

Pairwise comparisons (Mann–Whitney U):
Pair                                                   Statistic     p-value  Result
------------------------------------------------------------------------------------------
Lab, Trial 1: FHSS  vs Lab, Trial 2: FHSS                 54.000      0.0000  significant
Lab, Trial 1: FHSS  vs Lab, Trial 1: BPSK              26325.0

## 6. Plots

In [ ]:
# ── 6.1 PDR with Wilson CI ────────────────────────────────────────────────────
for label, m in metrics.items():
    n, k = m["n_sent"], m["n_matched"]
    lo, hi = proportion_confint(k, n, alpha=0.05, method="wilson")
    m["pdr_ci_err_lo"] = m["pdr"] * 100 - lo * 100
    m["pdr_ci_err_hi"] = hi * 100 - m["pdr"] * 100

short_labels_pdr = [
    "Lab T1<br>FHSS", "Lab T2<br>FHSS",
    "Lab T1<br>BPSK", "Lab T2<br>BPSK",
    "River T1<br>FHSS", "River T2<br>FHSS",
    "River T1<br>BPSK","River T2<br>BPSK",
]

fig = go.Figure()

for i, label in enumerate(labels):
    m = metrics[label]
    val = m["pdr"] * 100

    fig.add_trace(go.Bar(
        x=[short_labels_pdr[i]],
        y=[val],
        name=short_labels_pdr[i],
        marker_color=COLORS[i],
        marker_line_color="black",
        marker_line_width=0.8,
        text=[f"{val:.1f}%"],
        textposition="inside",
        insidetextanchor="middle",
        textfont=dict(size=11, color="black"),
        showlegend=False,
    ))

fig.add_vline(
    x=3.5,
    line_dash="dash",
    line_color="gray",
    line_width=1,
)

fig.add_annotation(
    x=1.5, y=108,
    text="<b>Laboratory</b>",
    showarrow=False,
    font=dict(size=16, color="gray"),
)

fig.add_annotation(
    x=5.5, y=108,
    text="<b>River</b>",
    showarrow=False,
    font=dict(size=16, color="gray"),
)

fig.update_layout(
    title=dict(
        text="Packet Delivery Rate (PDR)",
        font=dict(size=20),
        x=0.5,
        xanchor="center",
    ),
    width=680,
    height=420,
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Arial, sans-serif", size=16),
    margin=dict(l=55, r=20, t=55, b=60),
    bargap=0.30,
)

fig.update_yaxes(
    title_text="PDR (%)",
    title_font=dict(size=16),
    range=[0, 115],
    gridcolor="#e5e5e5",
    linecolor="black",
    mirror=True,
    dtick=10,
    tickfont=dict(size=14),
    tickvals=list(range(0, 101, 10)),
)

fig.update_xaxes(
    showgrid=False,
    linecolor="black",
    mirror=True,
    tickfont=dict(size=16),
)

save_fig(fig, "pdr")
fig.show()

NameError: name 'labels' is not defined

In [ ]:
# ── 6.2 Latency Violin with CI + operational thresholds ──────────────────────

short_labels_violin = [
    "Lab T1<br>FHSS", "Lab T2<br>FHSS",
    "Lab T1<br>BPSK", "Lab T2<br>BPSK",
    "River T1<br>FHSS", "River T2<br>FHSS",
    "River T1<br>BPSK","River T2<br>BPSK",
]

fig = go.Figure()

# ── Violin + CI traces ───────────────────────────────────────────────────────
for i, label in enumerate(labels):
    m  = metrics[label]
    sl = short_labels_violin[i]

    # Violin body
    fig.add_trace(go.Violin(
        y=m["latency_vals"],
        x=[sl] * len(m["latency_vals"]),
        name=sl.replace("<br>", " "),
        fillcolor=COLORS[i],
        opacity=0.55,
        line_color="rgba(0,0,0,0.5)",
        line_width=1,
        box_visible=True,
        box=dict(
            fillcolor="rgba(255,255,255,0.6)",
            line=dict(color="rgba(0,0,0,0.7)", width=2),
        ),
        meanline_visible=False,
        points=False,
        showlegend=False,
        width=0.8,
        spanmode="hard",
    ))

    # Mean + 95% CI
    fig.add_trace(go.Scatter(
        x=[sl],
        y=[m["lat_mean"]],
        mode="markers",
        marker=dict(
            symbol="diamond",
            color="black",
            size=8,
            line=dict(color="white", width=1),
        ),
        error_y=dict(
            type="data",
            symmetric=False,
            array=[m["ci_high"] - m["lat_mean"]],
            arrayminus=[m["lat_mean"] - m["ci_low"]],
            color="black",
            thickness=1.8,
            width=6,
        ),
        name="Mean ± 95% CI" if i == 0 else None,
        showlegend=(i == 0),
    ))

# ── Group separator ──────────────────────────────────────────────────────────
fig.add_vline(
    x=3.5,
    line_dash="dash",
    line_color="gray",
    line_width=1,
)


# ── Dynamic group label height (clean scaling) ───────────────────────────────
all_lat_max = max(m["lat_max"] for m in metrics.values())
group_label_y  = all_lat_max * 0.9

# ── Group annotations ────────────────────────────────────────────────────────
fig.add_annotation(
    x=1.5, y=group_label_y,
    text="<b>Laboratory</b>",
    showarrow=False,
    font=dict(size=16, color="gray"),
)
fig.add_annotation(
    x=5.5, y=group_label_y,
    text="<b>River</b>",
    showarrow=False,
    font=dict(size=16, color="gray"),
)

# ── Layout (matched style with PDR plot) ─────────────────────────────────────
fig.update_layout(
    title=dict(
        text="End-to-End Latency Distribution per Configuration",
        font=dict(size=20),
        x=0.5,
        xanchor="center",
    ),
    width=680,
    height=420,
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Arial, sans-serif", size=16),
    margin=dict(l=55, r=20, t=55, b=60),
    violingap=0.25,
    violingroupgap=0.1,
    legend=dict(
        x=0.01, y=0.99,
        bgcolor="rgba(255,255,255,0.9)",
        bordercolor="black",
        borderwidth=1,
    ),
)

# ── Axes styling (aligned with PDR style) ─────────────────────────────────────
fig.update_xaxes(
    showgrid=False,
    linecolor="black",
    mirror=True,
    tickfont=dict(size=16),
)

fig.update_yaxes(
    title_text="End-to-End Latency (ms)",
    title_font=dict(size=16),
    gridcolor="#e5e5e5",
    linecolor="black",
    mirror=True,
    dtick=1000,
    tickfont=dict(size=14),
)

save_fig(fig, "latency_violin")
fig.show()

  ✔ Saved: ./figures/latency_violin.pdf


In [ ]:
# ── 6.3 CDF of End-to-End Latency ────────────────────────────────────────────

def get_dash(label):
    return "solid" if "BPSK" in label else "dash"

fig = go.Figure()

# ── CDF traces ───────────────────────────────────────────────────────────────
for i, l in enumerate(labels):
    lat = np.sort(metrics[l]["latency_vals"])
    cdf = np.arange(1, len(lat) + 1) / len(lat)
    name = l.replace("\n", " ")

    fig.add_trace(go.Scatter(
        x=lat,
        y=cdf,
        mode="lines",
        name=name,
        line=dict(
            color=COLORS[i],
            width=2.2,
            dash=get_dash(l),
        ),
        hovertemplate=(
            f"<b>{name}</b><br>"
            "Latency: %{x:.0f} ms<br>"
            "CDF: %{y:.3f}<extra></extra>"
        ),
    ))

# ── Line-style legend entries (modulation encoding) ──────────────────────────
fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode="lines",
    line=dict(color="black", width=2.2, dash="solid"),
    name="BPSK",
    showlegend=True,
))
fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode="lines",
    line=dict(color="black", width=2.2, dash="dash"),
    name="FHSS",
    showlegend=True,
))

# ── Layout (matched with PDR + violin style) ──────────────────────────────────
fig.update_layout(
    title=dict(
        text="Cumulative Distribution Function of End-to-End Latency",
        font=dict(size=20),
        x=0.5,
        xanchor="center",
    ),
    width=680,
    height=420,
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Arial, sans-serif", size=16),
    margin=dict(l=55, r=20, t=55, b=60),
    legend=dict(
        x=0.60,
        y=0.35,
        bgcolor="rgba(255,255,255,0.92)",
        bordercolor="black",
        borderwidth=1,
        font=dict(size=12),
    ),
)

# ── Axes styling (consistent frame + grid) ────────────────────────────────────
fig.update_xaxes(
    title_text="End-to-End Latency (ms)",
    showgrid=False,
    linecolor="black",
    mirror=True,
    tickfont=dict(size=14),
    dtick=500,
)

fig.update_yaxes(
    title_text="Cumulative Probability",
    gridcolor="#e5e5e5",
    linecolor="black",
    mirror=True,
    range=[0, 1.05],
    dtick=0.1,
    tickformat=".1f",
    tickfont=dict(size=14),
)

save_fig(fig, "latency_cdf")
fig.show()

  ✔ Saved: ./figures/latency_cdf.pdf


In [ ]:
# ── 6.4 Jitter Violin ────────────────────────────────────────────────────────

short_labels_jitter = [
    "Lab T1<br>FHSS", "Lab T2<br>FHSS",
    "Lab T1<br>BPSK", "Lab T2<br>BPSK",
    "River T1<br>FHSS", "River T2<br>FHSS",
    "River T1<br>BPSK","River T2<br>BPSK",
]

fig = go.Figure()

# ── Violin + mean markers ────────────────────────────────────────────────────
for i, l in enumerate(labels):
    m  = metrics[l]
    sl = short_labels_jitter[i]

    fig.add_trace(go.Violin(
        y=m["jitter_vals"],
        x=[sl] * len(m["jitter_vals"]),
        name=sl.replace("<br>", " "),
        fillcolor=COLORS[i],
        opacity=0.55,
        line_color="rgba(0,0,0,0.5)",
        line_width=1,
        box_visible=True,
        box=dict(
            fillcolor="rgba(255,255,255,0.6)",
            line=dict(color="rgba(0,0,0,0.7)", width=2),
        ),
        meanline_visible=False,
        points=False,
        showlegend=False,
        width=0.8,
        spanmode="hard",
    ))

# ── Group separator ───────────────────────────────────────────────────────────
fig.add_vline(
    x=3.5,
    line_dash="dash",
    line_color="gray",
    line_width=1,
)

# ── Dynamic group label height (clean scaling) ───────────────────────────────
all_max_jitter = max(m["jitter_max"] for m in metrics.values())
group_label_y  = all_max_jitter * 1.08

fig.add_annotation(
    x=1.5,
    y=group_label_y,
    text="<b>Laboratory</b>",
    showarrow=False,
    font=dict(size=16, color="gray"),
)

fig.add_annotation(
    x=5.5,
    y=group_label_y,
    text="<b>River</b>",
    showarrow=False,
    font=dict(size=16, color="gray"),
)

# ── Layout (fully aligned with other figures) ────────────────────────────────
fig.update_layout(
    title=dict(
        text="Jitter Distribution per Configuration",
        font=dict(size=20),
        x=0.5,
        xanchor="center",
    ),
    width=680,
    height=420,
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Arial, sans-serif", size=16),
    margin=dict(l=55, r=20, t=55, b=60),
    violingap=0.25,
    violingroupgap=0.1,
    legend=dict(
        x=0.01,
        y=0.99,
        bgcolor="rgba(255,255,255,0.9)",
        bordercolor="black",
        borderwidth=1,
        font=dict(size=12),
    ),
)

# ── Axes styling (consistent frame + grid) ───────────────────────────────────
fig.update_xaxes(
    showgrid=False,
    linecolor="black",
    mirror=True,
    tickfont=dict(size=14),
)

fig.update_yaxes(
    title_text="Jitter (ms)",
    gridcolor="#e5e5e5",
    linecolor="black",
    mirror=True,
    tickfont=dict(size=14),
)

save_fig(fig, "jitter_violin")
fig.show()

  ✔ Saved: ./figures/jitter_violin.pdf


In [ ]:
import math
import pandas as pd
from plotly.subplots import make_subplots
import plotly.graph_objects as go


def nice_dtick(values, target_ticks=6):
    vmin, vmax = min(values), max(values)
    span = vmax - vmin if vmax > vmin else 1
    raw  = span / target_ticks
    mag  = 10 ** math.floor(math.log10(raw))
    norm = raw / mag
    if   norm < 1.5: nice = 1
    elif norm < 3:   nice = 2
    elif norm < 7:   nice = 5
    else:            nice = 10
    return nice * mag


fig = make_subplots(
    rows=len(labels), cols=1,
    shared_xaxes=False,
    subplot_titles=[l.replace("\n", " ") for l in labels],
    vertical_spacing=0.03,
)

for i, l in enumerate(labels, start=1):
    m = metrics[l]

    # ── Reconstruct full x-axis (original packet indices) ────────────────────
    full_len      = m["lat_full_len"]
    full_x        = np.arange(full_len)

    # Cleaned latency placed at their original positions
    outlier_idx   = set(m["outlier_indices"])
    clean_indices = [j for j in range(full_len) if j not in outlier_idx]
    lat_clean     = m["latency_vals"]                   # already cleaned array

    # Outlier positions and values
    out_x         = m["outlier_indices"]
    out_y         = m["outlier_latency_vals_ms"]

    # Moving average on cleaned values only (indexed at clean positions)
    window      = 10
    moving_avg  = (
        pd.Series(lat_clean)
        .rolling(window=window, center=True, min_periods=1)
        .mean()
        .values
    )

    upper = np.full(len(clean_indices), m["lat_mean"] + m["lat_std"])
    lower = np.full(len(clean_indices), m["lat_mean"] - m["lat_std"])

    cx = np.array(clean_indices)

    # ── Raw latency (cleaned) ─────────────────────────────────────────────────
    fig.add_trace(go.Scatter(
        x=cx, y=lat_clean,
        mode="lines",
        line=dict(color=COLORS[i - 1], width=1.4),
        opacity=0.85,
        name=l.replace("\n", " "),
        showlegend=False,
    ), row=i, col=1)

    # ── Moving average ────────────────────────────────────────────────────────
    fig.add_trace(go.Scatter(
        x=cx, y=moving_avg,
        mode="lines",
        line=dict(color="black", width=2.2),
        opacity=0.7,
        name="Moving avg (w=10)" if i == 1 else None,
        showlegend=(i == 1),
        legendgroup="mavg",
    ), row=i, col=1)

    # ── Mean ± σ band ─────────────────────────────────────────────────────────
    fig.add_trace(go.Scatter(
        x=np.concatenate([cx, cx[::-1]]),
        y=np.concatenate([upper, lower[::-1]]),
        fill="toself",
        fillcolor=(
            f"rgba("
            f"{int(COLORS[i-1][1:3], 16)},"
            f"{int(COLORS[i-1][3:5], 16)},"
            f"{int(COLORS[i-1][5:7], 16)},"
            f"0.12)"
        ),
        line=dict(color="rgba(0,0,0,0)"),
        name="Mean ± σ" if i == 1 else None,
        showlegend=(i == 1),
        legendgroup="std",
    ), row=i, col=1)

    # ── Mean line ─────────────────────────────────────────────────────────────
    fig.add_trace(go.Scatter(
        x=[0, full_len - 1],
        y=[m["lat_mean"], m["lat_mean"]],
        mode="lines",
        line=dict(color="red", dash="dash", width=1.2),
        name="Mean" if i == 1 else None,
        showlegend=(i == 1),
        legendgroup="mean",
    ), row=i, col=1)

    # ── Median line ───────────────────────────────────────────────────────────
    fig.add_trace(go.Scatter(
        x=[0, full_len - 1],
        y=[m["lat_median"], m["lat_median"]],
        mode="lines",
        line=dict(color="orange", dash="dot", width=1.2),
        name="Median" if i == 1 else None,
        showlegend=(i == 1),
        legendgroup="median",
    ), row=i, col=1)

    # ── Outlier threshold line ────────────────────────────────────────────────
    if len(out_x) > 0:
        fig.add_trace(go.Scatter(
            x=[0, full_len - 1],
            y=[LATENCY_OUTLIER_MS, LATENCY_OUTLIER_MS],
            mode="lines",
            line=dict(color="crimson", dash="longdash", width=1.0),
            opacity=0.5,
            name=f"Outlier threshold ({LATENCY_OUTLIER_MS//1000} s)" if i == 1 else None,
            showlegend=(i == 1),
            legendgroup="thresh",
        ), row=i, col=1)

        # ── Outlier markers (X) ───────────────────────────────────────────────
        fig.add_trace(go.Scatter(
            x=out_x,
            y=out_y,
            mode="markers",
            marker=dict(
                symbol="x",
                size=7,
                color="crimson",
                line=dict(width=2),
            ),
            name=f"Outliers (>{LATENCY_OUTLIER_MS//1000} s)" if i == 1 else None,
            showlegend=(i == 1),
            legendgroup="outliers",
            hovertemplate="Sample %{x}<br>Latency: %{y:.0f} ms<extra></extra>",
        ), row=i, col=1)

    # ── Axes ──────────────────────────────────────────────────────────────────
    fig.update_yaxes(
        title_text="Latency (ms)",
        title_font=dict(size=20),
        gridcolor="#e5e5e5",
        linecolor="black",
        mirror=True,
        dtick=nice_dtick(
            np.concatenate([lat_clean, out_y]) if len(out_y) > 0 else lat_clean
        ),
        tickfont=dict(size=18),
        row=i, col=1,
    )
    fig.update_xaxes(
        title_text="Sample index" if i == len(labels) else None,
        showgrid=False,
        linecolor="black",
        mirror=True,
        tickfont=dict(size=18),
        range=[-1, full_len],
        row=i, col=1,
    )

# ── Global layout ─────────────────────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text="Latency Evolution Over Time per Configuration",
        font=dict(size=20),
        x=0.5, xanchor="center",
    ),
    height=300 * len(labels),
    width=950,
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Arial, sans-serif", size=14),
    margin=dict(l=70, r=40, t=60, b=60),
    legend=dict(
        x=0.72, y=0.99,
        bgcolor="rgba(255,255,255,0.92)",
        bordercolor="black",
        borderwidth=1,
        font=dict(size=12),
        tracegroupgap=3,
    ),
)

fig.update_annotations(font=dict(size=22))

save_fig(fig, "latency_time")
fig.show()

  ✔ Saved: ./figures/latency_time.pdf


In [ ]:
# ── 6.6 Median Latency with 95% CI ───────────────────────────────────────────

short_labels_bar = [
    "Lab T1<br>FHSS", "Lab T2<br>FHSS",
    "Lab T1<br>BPSK", "Lab T2<br>BPSK",
    "River T1<br>FHSS", "River T2<br>FHSS",
    "River T1<br>BPSK","River T2<br>BPSK",
]

medians = [metrics[l]["lat_median"] for l in labels]

ci_lo = [
    metrics[l]["lat_median"] - metrics[l]["lat_median_ci_low"]
    for l in labels
]

ci_hi = [
    metrics[l]["lat_median_ci_high"] - metrics[l]["lat_median"]
    for l in labels
]

fig = go.Figure()

# ── Bars with CI ──────────────────────────────────────────────────────────────
for i, (l, median, lo, hi) in enumerate(zip(labels, medians, ci_lo, ci_hi)):

    fig.add_trace(go.Bar(
        x=[short_labels_bar[i]],
        y=[median],
        name=l.replace("\n", " "),
        marker_color=COLORS[i],
        marker_line_color="black",
        marker_line_width=0.8,
        error_y=dict(
            type="data",
            symmetric=False,
            array=[hi],
            arrayminus=[lo],
            color="black",
            thickness=1.5,
            width=6,
        ),
        showlegend=False,
    ))

    # Value label (clean, readable, consistent placement)
    fig.add_annotation(
        x=short_labels_bar[i],
        y=median * 0.95,
        text=f"<b>{median:.0f}</b>",
        showarrow=False,
        font=dict(size=12, color="white"),
    )

# ── Group separator ───────────────────────────────────────────────────────────
fig.add_vline(
    x=3.5,
    line_dash="dash",
    line_color="gray",
    line_width=1,
)

# ── Dynamic group labels ──────────────────────────────────────────────────────
y_top = max(medians) * 1.18

fig.add_annotation(
    x=1.5,
    y=y_top,
    text="<b>Laboratory</b>",
    showarrow=False,
    font=dict(size=16, color="gray"),
)

fig.add_annotation(
    x=5.5,
    y=y_top,
    text="<b>River</b>",
    showarrow=False,
    font=dict(size=16, color="gray"),
)

# ── Layout (fully aligned with PDR + other figures) ──────────────────────────
fig.update_layout(
    title=dict(
        text="Median End-to-End Latency with 95% CI",
        font=dict(size=20),
        x=0.5,
        xanchor="center",
    ),
    width=680,
    height=420,
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Arial, sans-serif", size=16),
    margin=dict(l=55, r=20, t=55, b=60),
    bargap=0.35,
    showlegend=True,
    legend=dict(
        x=0.70,
        y=0.98,
        bgcolor="rgba(255,255,255,0.92)",
        bordercolor="black",
        borderwidth=1,
        font=dict(size=12),
    ),
)

# ── Axes styling (consistent frame + grid) ───────────────────────────────────
fig.update_xaxes(
    showgrid=False,
    linecolor="black",
    mirror=True,
    tickfont=dict(size=14),
)

fig.update_yaxes(
    title_text="Median Latency (ms)",
    gridcolor="#e5e5e5",
    linecolor="black",
    mirror=True,
    dtick=500,
    range=[0, max(medians) * 1.30],
    tickfont=dict(size=14),
)

save_fig(fig, "latency_median_ci")
fig.show()

  ✔ Saved: ./figures/latency_median_ci.pdf
